In [9]:
import pandas as pd
import numpy as np
import sklearn.preprocessing as sks
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg



In [10]:
data=pd.read_csv("insurance.csv")
data
sencoder=sks.OneHotEncoder(sparse_output=False,handle_unknown="ignore")
scaler=sks.StandardScaler()
lr=LinearRegression()
sg=SGDRegressor()
num_cols=['charges','bmi','age']


In [11]:
def process_data(df):
 
    region_encoded = sencoder.fit_transform(df[['region']])
    encoded_cols = sencoder.get_feature_names_out(['region'])
    encoded_df = pd.DataFrame(region_encoded, columns=encoded_cols, index=df.index)
    df = pd.concat([df.drop(columns='region'), encoded_df], axis=1)

    sex_encoded = sencoder.fit_transform(df[['sex']])
    encoded_cols = sencoder.get_feature_names_out(['sex'])
    encoded_df = pd.DataFrame(sex_encoded, columns=encoded_cols, index=df.index)
    df = pd.concat([df.drop(columns='sex'), encoded_df], axis=1)

    smoker_encoded = sencoder.fit_transform(df[['smoker']])
    encoded_cols = sencoder.get_feature_names_out(['smoker'])
    encoded_df = pd.DataFrame(smoker_encoded, columns=encoded_cols, index=df.index)
    df = pd.concat([df.drop(columns='smoker'), encoded_df], axis=1)

    df[num_cols]=scaler.fit_transform(df[num_cols])
    return df

new_data=process_data(data)
x = new_data.drop(['charges'], axis=1)
y = new_data['charges']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)


In [12]:
def normal():
    lr = LinearRegression()
    lr.fit(x_train,y_train)
    y_pred = lr.predict(x_test)
    print(f"Linear Regression R2: {lr.score(x_test, y_test)}")
    params={
        'penalty':[ 'l2' ,'l1', 'elasticnet', None],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate': ['invscaling', 'adaptive'],
        'eta0': [0.01, 0.05]
    }
    sg=SGDRegressor()
    sgd_grid = GridSearchCV(estimator=sg, param_grid=params, cv=5, scoring='r2')
    sgd_grid.fit(x_train, y_train)
    print(f"SGD Score: {sgd_grid.best_score_}")
    dt = DecisionTreeRegressor(random_state=42)
    dt.fit(x_train, y_train)
    print(f"Decision Tree R2: {dt.score(x_test, y_test)}")
normal()


Linear Regression R2: 0.7999876970680433
SGD Score: 0.7311116122094683
Decision Tree R2: 0.6687575781765068


In [13]:
def ensemble():
    param = {
        'n_estimators': [100, 500, 1000],
        'max_features': ['auto', 'sqrt', 'log2', None],
        'bootstrap': [True, False],
        'max_depth': [10, 11, 110, None],
    }
    rf = RandomForestRegressor(oob_score=True,n_jobs=-1)
    rf_grid = GridSearchCV(estimator=rf, param_grid=param, cv=5, scoring='r2')
    rf_grid.fit(x_train, y_train)
    print(f"Random Forest best CV r2: {rf_grid.best_score_}, best params: {rf_grid.best_params_},")
    parameters={
        'n_estimators': [100, 500, 1000],
        'max_depth': [3, 6, 9, None],
        'learning_rate':[0.01,0.05,0.1,0.2],
    }
    mod=xg.XGBRegressor(eval_metric='logloss')
    xg_grid=GridSearchCV(estimator=mod,param_grid=parameters,cv=5,scoring='r2')
    xg_grid.fit(x_train,y_train)
    print(f"XGBoost best CV r2:{xg_grid.best_score_},best params:{xg_grid.best_params_}")
ensemble()


g:\Project\complete-pandas-tutorial\venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
300 fits failed out of a total of 480.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
120 fits failed with the following error:
Traceback (most recent call last):
  File "g:\Project\complete-pandas-tutorial\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\Project\complete-pandas-tutorial\venv\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "g:\Project\complete-pandas-tutorial\venv\Lib

Random Forest best CV r2: 0.8319306593368513, best params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'log2', 'n_estimators': 1000},
XGBoost best CV r2:0.8490324088198274,best params:{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500}


Why use XGBoost over other boosting/bagging techniques

1.When Random Forest was applied on the dataset, it showed an r2 score of around 0.83 while XGBoost clearly showed as r2 score of 0.85 while Linear Regression, Decision Trees and SGD Regressor showed r2 scores less than 0.8-0.7.


2.When compared to techniques like Adaboost and Gradient Boosting, XGBoost is much more advanced and robust and can be used to speed up the learning process, as it works on parallel processing.


3.XGBoost is just a special and more refined form of gradient boosting(Extreme Gradient boosting=XGBoost)


4.Rather than bagging techniques like Random Forests, where models take decision based on majority count of the estimators(in this case the average of the output of each estimator), using XGBoost where the current model learns from the mistakes of the previous model, makes the model(in my opinion)much more accurate.


5.Since the data has many outliers in the charges column, XGBoost has the loss function whose derivative is used to minimize the residual error. So this algorithm can work with these outliers much more effectively.


6.Binary columns like smoker_yes and sex_male might be used as a Gain feature by XGBoost by creating splits based on these features.


7.Hyperparameter tuning helps in controlling factors like overfitting(by reducing max_depth),by allowing the algorithm to take smaller jumps in the weights assigned to each feature(by controlling learning_rate).